# Lesson 3: Custom Tools

## WHY
An LLM can *think*, but it can't *do* anything on its own — it can't check a database, call an API, or mute a Discord user.
**Tools** bridge that gap. A tool is just a Python function with a description that tells the LLM *what it does* and *what arguments it needs*. When the LLM decides it needs to take an action, it "calls" the tool, your code runs it, and the result goes back to the LLM.

In our moderation bot, tools will eventually let the agent:
- Look up server rules
- Check a user's warn history
- Issue a warning or timeout
- Search a FAQ knowledge base

**By the end of this notebook you will:**
1. Understand what a "tool" is in the LangChain ecosystem
2. Create tools with the `@tool` decorator (simplest approach)
3. Define tool input schemas with Pydantic for validation and documentation
4. Bind tools to a chat model with `.bind_tools()`
5. Interpret the `AIMessage.tool_calls` response
6. Execute tool calls manually to understand the full loop
7. Build moderation-specific tools ready for the bot

## Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
print("API key loaded:", "OPENAI_API_KEY" in os.environ)

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## WHAT — Tools in LangChain

A **tool** is a Python function that the LLM can request to call. LangChain tools have three pieces:

| Piece | What it is | Example |
|-------|-----------|--------|
| **Name** | A short identifier the LLM uses to pick the tool | `"get_server_rules"` |
| **Description** | Natural language explaining when/why to use it | `"Look up the rules for a Discord server"` |
| **Schema** | The arguments the tool accepts (types, descriptions) | `server_id: str` |

The LLM **never runs your function directly**. It returns a structured "please call this tool with these args" message. Your code is responsible for actually executing it. This is a critical safety boundary — you control what happens.

📖 [Tools conceptual guide](https://python.langchain.com/docs/concepts/tools/)

## HOW — Creating Tools with `@tool`

The `@tool` decorator is the simplest way to create a tool. It reads:
- The **function name** → becomes the tool name
- The **docstring** → becomes the tool description (the LLM reads this!)
- The **type hints** → becomes the argument schema

This means your docstring is essentially a prompt to the LLM about when to use this tool.

In [ ]:
from langchain_core.tools import tool


@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a piece of text.

    Use this when you need to analyze message length.
    """
    return len(text.split())


# Inspect what LangChain created
print(f"Name:        {get_word_count.name}")
print(f"Description: {get_word_count.description}")
print(f"Schema:      {get_word_count.args_schema.model_json_schema()}")

In [ ]:
# You can call a tool directly — it's still just a Python function
result = get_word_count.invoke({"text": "Hello everyone how are you"})
print(f"Word count: {result}")

### Multiple Parameters

Tools can take multiple arguments. The LLM will fill in each one based on the schema.

In [ ]:
@tool
def check_message_contains(message: str, keyword: str) -> bool:
    """Check if a Discord message contains a specific keyword (case-insensitive).

    Use this to scan messages for banned words or patterns.
    """
    return keyword.lower() in message.lower()


# Direct call to verify behavior
result = check_message_contains.invoke({"message": "Check out my FREE giveaway!", "keyword": "free"})
print(f"Contains keyword: {result}")

## WHAT — Pydantic Schemas for Tool Inputs

The `@tool` decorator auto-generates a schema from type hints, but for production tools you often want:
- **Field descriptions** that help the LLM understand each argument
- **Validation** (e.g., max length, allowed values)
- **Default values** for optional arguments

You can provide a Pydantic `BaseModel` as the `args_schema` to get all of this.
The LLM reads field descriptions just like it reads the tool description — they're part of the prompt.

📖 [Pydantic docs](https://docs.pydantic.dev/latest/)

In [ ]:
from pydantic import BaseModel, Field


class ModerationCheckInput(BaseModel):
    """Input schema for the moderation check tool."""

    message: str = Field(
        description="The Discord message text to check for rule violations"
    )
    rule_category: str = Field(
        default="general",
        description="Which rule category to check against: general, spam, nsfw, or harassment"
    )


@tool(args_schema=ModerationCheckInput)
def check_moderation(message: str, rule_category: str = "general") -> str:
    """Check if a message violates Discord server rules.

    Use this tool when you need to evaluate whether a user's message
    breaks any server rules. Returns a verdict.
    """
    # Simulated logic — in production this could query a database or call another API
    spam_keywords = ["free", "giveaway", "click here", "buy now", "discount"]

    if rule_category == "spam":
        is_spam = any(kw in message.lower() for kw in spam_keywords)
        return f"SPAM DETECTED: {is_spam}"
    elif rule_category == "harassment":
        # Placeholder — a real implementation would use content classification
        return "HARASSMENT CHECK: No violations detected (placeholder)"
    else:
        return f"GENERAL CHECK: Message appears to follow {rule_category} rules"


# Inspect the richer schema
import json
print(json.dumps(check_moderation.args_schema.model_json_schema(), indent=2))

In [ ]:
# Test with explicit args
result = check_moderation.invoke({"message": "FREE NITRO GIVEAWAY IN DMS!", "rule_category": "spam"})
print(result)

# Test with default rule_category
result = check_moderation.invoke({"message": "Hello everyone!"})
print(result)

## HOW — Binding Tools to a Model

Creating a tool doesn't magically connect it to the LLM. You need to **bind** tools to the model with `.bind_tools()`.
This tells the model:
1. These tools exist
2. Here's what each one does (name, description, schema)
3. You may request to call them in your response

The model returns an `AIMessage` with a `.tool_calls` list — each entry says which tool to call and with what arguments.

In [ ]:
# Bind tools to the model
tools = [get_word_count, check_message_contains, check_moderation]
llm_with_tools = llm.bind_tools(tools)

print(f"Model now knows about {len(tools)} tools")
print(f"Type: {type(llm_with_tools)}")

In [ ]:
from langchain_core.messages import HumanMessage

# Ask the model something that should trigger a tool call
response = llm_with_tools.invoke([
    HumanMessage(content="Check if this message is spam: 'FREE DISCORD NITRO CLICK HERE NOW'")
])

# The model didn't run the tool — it REQUESTED a tool call
print("=== Content (may be empty when tool call is made) ===")
print(repr(response.content))

print("\n=== Tool Calls ===")
for tc in response.tool_calls:
    print(f"  Tool: {tc['name']}")
    print(f"  Args: {tc['args']}")
    print(f"  ID:   {tc['id']}")

### Understanding Tool Calls

Notice what happened above:
- The model's `.content` is empty (or minimal) — it decided to use a tool instead of answering directly
- `.tool_calls` contains structured data: **which** tool, **what arguments**, and a **unique ID**
- The model chose the tool and filled in the arguments based on your message

The model does NOT run the tool. It's a *request*. Your code decides whether to actually execute it.
This is the safety layer that makes AI tools production-viable.

In [ ]:
# What if the model doesn't need a tool? It just responds normally.
response = llm_with_tools.invoke([
    HumanMessage(content="What is a Discord server?")
])

print(f"Content: {response.content[:100]}...")
print(f"Tool calls: {response.tool_calls}")
print("\n→ No tool calls! The model answered directly.")

## HOW — Executing Tool Calls (The Full Loop)

When the model requests a tool call, you need to:
1. Extract the tool name and arguments from `AIMessage.tool_calls`
2. Find and run the actual tool function
3. Send the result back as a `ToolMessage`
4. Let the model generate a final response using the tool result

This is the **tool execution loop** — the core pattern behind every LangChain agent.
In Lesson 4, `create_react_agent` automates this loop, but here we'll do it manually so you understand every step.

In [ ]:
from langchain_core.messages import HumanMessage, ToolMessage

# Step 1: Ask the model something that requires a tool
messages = [
    HumanMessage(content="How many words are in this message: 'The quick brown fox jumps over the lazy dog'")
]
ai_response = llm_with_tools.invoke(messages)

print("Step 1 — Model's response:")
print(f"  Content: {repr(ai_response.content)}")
print(f"  Tool calls: {ai_response.tool_calls}")

In [ ]:
# Step 2: Build a lookup table so we can find tools by name
tool_map = {t.name: t for t in tools}
print(f"Available tools: {list(tool_map.keys())}")

# Step 3: Execute each tool call and collect ToolMessages
messages.append(ai_response)  # add the AI's tool-call message to history

for tool_call in ai_response.tool_calls:
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    tool_id = tool_call["id"]

    print(f"\nExecuting: {tool_name}({tool_args})")
    result = tool_map[tool_name].invoke(tool_args)
    print(f"Result: {result}")

    # Wrap result as ToolMessage — the tool_call_id links it back
    messages.append(ToolMessage(content=str(result), tool_call_id=tool_id))

In [ ]:
# Step 4: Send the full conversation (including tool results) back to the model
final_response = llm_with_tools.invoke(messages)

print("Step 4 — Final response:")
print(final_response.content)
print(f"\nTool calls in final response: {final_response.tool_calls}")
print("→ No more tool calls — the model used the result to give a final answer.")

### The Full Conversation Flow

Let's visualize all the messages that were exchanged:

```
1. HumanMessage    → "How many words in..."     (you)
2. AIMessage       → tool_calls: [get_word_count({text: ...})]  (model decides to use tool)
3. ToolMessage     → "9"                        (your code ran the tool)
4. AIMessage       → "The message contains 9 words"  (model uses result)
```

This 4-step loop is what every agent does, over and over. The only difference is that agents automate steps 2-3 in a loop until the model stops requesting tools.

## HOW — Building Moderation Tools

Now let's build tools that are actually useful for our Discord moderation bot.
These are designed to be **importable** — you'll copy them into the bot codebase later.

For now, they use simulated data. In later modules, they'll connect to real Discord APIs and databases.

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field


# ── Tool 1: Lookup server rules ────────────────────────────────

# Simulated rules database
SERVER_RULES = {
    "general": [
        "Be respectful to all members",
        "Keep discussions on-topic in each channel",
        "No excessive caps or unicode spam",
    ],
    "spam": [
        "No unsolicited advertisements or self-promotion",
        "No repeated messages or copy-paste spam",
        "No phishing links or scam giveaways",
    ],
    "nsfw": [
        "No NSFW content outside of designated channels",
        "No gore or shock content anywhere",
    ],
}


@tool
def get_server_rules(category: str = "all") -> str:
    """Look up the Discord server's moderation rules.

    Use this tool when you need to check what rules apply before making
    a moderation decision. Returns rules for the given category.

    Parameters
    ----------
    category : str
        Rule category to look up: 'general', 'spam', 'nsfw', or 'all'.
    """
    if category == "all":
        lines = []
        for cat, rules in SERVER_RULES.items():
            lines.append(f"\n## {cat.upper()}")
            for i, rule in enumerate(rules, 1):
                lines.append(f"  {i}. {rule}")
        return "\n".join(lines)

    if category in SERVER_RULES:
        rules = SERVER_RULES[category]
        return "\n".join(f"{i}. {r}" for i, r in enumerate(rules, 1))

    return f"Unknown category: {category}. Available: {list(SERVER_RULES.keys())}"

In [ ]:
# Test the rules tool
print(get_server_rules.invoke({"category": "spam"}))
print("\n--- All rules ---")
print(get_server_rules.invoke({"category": "all"}))

In [ ]:
# ── Tool 2: Lookup user warn history ────────────────────────────

# Simulated warn database
WARN_HISTORY = {
    "user_123": [
        {"reason": "Spam links in #general", "date": "2026-03-15"},
        {"reason": "Harassment toward another member", "date": "2026-04-01"},
    ],
    "user_456": [
        {"reason": "NSFW content in wrong channel", "date": "2026-02-20"},
    ],
    "user_789": [],
}


class UserLookupInput(BaseModel):
    """Input for looking up a user's moderation history."""

    user_id: str = Field(description="The Discord user's ID to look up")


@tool(args_schema=UserLookupInput)
def get_user_warn_history(user_id: str) -> str:
    """Look up a Discord user's previous warnings and infractions.

    Use this tool to check a user's history before deciding on moderation
    action. Repeat offenders may need stricter action.
    """
    if user_id not in WARN_HISTORY:
        return f"No record found for user {user_id}"

    warnings = WARN_HISTORY[user_id]
    if not warnings:
        return f"User {user_id} has a clean record (0 warnings)"

    lines = [f"User {user_id} has {len(warnings)} warning(s):"]
    for w in warnings:
        lines.append(f"  - {w['date']}: {w['reason']}")
    return "\n".join(lines)

In [ ]:
# Test the history tool
print(get_user_warn_history.invoke({"user_id": "user_123"}))
print()
print(get_user_warn_history.invoke({"user_id": "user_789"}))

In [ ]:
# ── Tool 3: Issue moderation action ────────────────────────────

class ModActionInput(BaseModel):
    """Input for issuing a moderation action."""

    user_id: str = Field(description="The Discord user ID to moderate")
    action: str = Field(
        description="The moderation action to take: 'warn', 'mute', 'kick', or 'ban'"
    )
    reason: str = Field(description="Explanation of why this action is being taken")
    duration_minutes: int = Field(
        default=0,
        description="Duration in minutes for temporary actions like mute. 0 = permanent."
    )


@tool(args_schema=ModActionInput)
def issue_moderation_action(
    user_id: str, action: str, reason: str, duration_minutes: int = 0
) -> str:
    """Issue a moderation action against a Discord user.

    Use this tool to enforce server rules. Always check user history
    and server rules BEFORE issuing an action.

    IMPORTANT: This tool simulates the action for safety. In production,
    it would call the Discord API.
    """
    valid_actions = ["warn", "mute", "kick", "ban"]
    if action not in valid_actions:
        return f"Invalid action '{action}'. Must be one of: {valid_actions}"

    # Simulate — in production this would call Discord's API via py-cord
    duration_str = f" for {duration_minutes} minutes" if duration_minutes > 0 else ""
    return (
        f"[SIMULATED] Action: {action.upper()}{duration_str}\n"
        f"User: {user_id}\n"
        f"Reason: {reason}"
    )

In [ ]:
# Test the moderation action tool
result = issue_moderation_action.invoke({
    "user_id": "user_123",
    "action": "mute",
    "reason": "Repeated spam after warning",
    "duration_minutes": 30,
})
print(result)

## HOW — Full Tool Loop with Moderation Tools

Let's put all three moderation tools together and watch the model use them to handle a moderation scenario.

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

# Bind all moderation tools
moderation_tools = [get_server_rules, get_user_warn_history, issue_moderation_action]
mod_llm = llm.bind_tools(moderation_tools)
tool_map = {t.name: t for t in moderation_tools}

# Set up the conversation
messages = [
    SystemMessage(
        content=(
            "You are a Discord server moderator bot.\n"
            "When evaluating a message, ALWAYS:\n"
            "1. Check the relevant server rules first\n"
            "2. Check the user's warning history\n"
            "3. Then decide on an appropriate action\n"
            "Be fair but firm. Repeat offenders get stricter action."
        )
    ),
    HumanMessage(
        content=(
            "User user_123 just posted this message in #general: "
            "'FREE DISCORD NITRO! Click this link: totally-not-a-scam.com'\n"
            "Please evaluate and take appropriate action."
        )
    ),
]

print("=== Starting moderation loop ===")
print(f"Messages so far: {len(messages)}\n")

# Run the tool loop — keep going until the model stops requesting tools
max_iterations = 10  # safety limit
iteration = 0

while iteration < max_iterations:
    iteration += 1
    print(f"--- Iteration {iteration} ---")

    ai_response = mod_llm.invoke(messages)
    messages.append(ai_response)

    # If no tool calls, the model is done
    if not ai_response.tool_calls:
        print("Model gave final response (no tool calls)")
        break

    # Execute each tool call
    for tc in ai_response.tool_calls:
        t_name = tc["name"]
        t_args = tc["args"]
        print(f"  Calling: {t_name}({t_args})")
        res = tool_map[t_name].invoke(t_args)
        res_str = str(res)
        display = res_str[:80] + "..." if len(res_str) > 80 else res_str
        print(f"  Result: {display}")
        messages.append(ToolMessage(content=res_str, tool_call_id=tc["id"]))
    print()

print(f"\n=== Final verdict (after {iteration} iterations) ===")
print(ai_response.content)

## Deep Dive — How Tool Schemas Become Part of the Prompt

When you call `.bind_tools()`, LangChain converts each tool's schema into a JSON specification that gets sent as part of the API request.
The LLM sees something like this (simplified):

```json
{
  "name": "get_server_rules",
  "description": "Look up the Discord server's moderation rules...",
  "parameters": {
    "type": "object",
    "properties": {
      "category": {
        "type": "string",
        "description": "Rule category to look up: 'general', 'spam', 'nsfw', or 'all'"
      }
    }
  }
}
```

This is why **good descriptions matter** — they're literally part of the prompt.
A vague description like `"Does stuff"` will confuse the LLM about when to use the tool.
A specific description like `"Look up the Discord server's moderation rules. Use this when you need to check what rules apply before making a moderation decision."` tells the LLM exactly when it's appropriate.

## Importable Moderation Tools

Here's the full set of tools packaged for import.
The simulated data (`SERVER_RULES`, `WARN_HISTORY`) would be replaced with database/API calls in production.

In [ ]:
from langchain_core.tools import tool
from pydantic import BaseModel, Field


# ── Schemas ─────────────────────────────────────────────────────

class UserLookupInput(BaseModel):
    """Input for user history lookup."""

    user_id: str = Field(description="The Discord user's ID")


class ModActionInput(BaseModel):
    """Input for moderation actions."""

    user_id: str = Field(description="Discord user ID to moderate")
    action: str = Field(description="Action: 'warn', 'mute', 'kick', or 'ban'")
    reason: str = Field(description="Reason for the action")
    duration_minutes: int = Field(default=0, description="Duration for temp actions. 0 = permanent.")


# ── Tools ───────────────────────────────────────────────────────

@tool
def get_server_rules(category: str = "all") -> str:
    """Look up the Discord server's moderation rules.

    Use this when you need to check what rules apply before making
    a moderation decision. Returns rules for the given category.
    """
    # Replace with real data source in production
    rules = {"general": ["Be respectful", "Stay on-topic", "No excessive caps"],
             "spam": ["No self-promo", "No repeated messages", "No phishing"],
             "nsfw": ["No NSFW outside designated channels", "No gore"]}
    if category == "all":
        return "\n".join(f"{c}: {', '.join(r)}" for c, r in rules.items())
    return ", ".join(rules.get(category, [f"Unknown category: {category}"]))


@tool(args_schema=UserLookupInput)
def get_user_warn_history(user_id: str) -> str:
    """Look up a user's previous warnings.

    Check history before deciding on action. Repeat offenders need stricter action.
    """
    # Replace with database lookup in production
    return f"User {user_id}: 0 previous warnings (simulated)"


@tool(args_schema=ModActionInput)
def issue_moderation_action(
    user_id: str, action: str, reason: str, duration_minutes: int = 0
) -> str:
    """Issue a moderation action against a Discord user.

    Always check rules and history BEFORE calling this.
    Simulated in dev — calls Discord API in production.
    """
    valid = ["warn", "mute", "kick", "ban"]
    if action not in valid:
        return f"Invalid action. Must be one of: {valid}"
    dur = f" for {duration_minutes}m" if duration_minutes else ""
    return f"[SIM] {action.upper()}{dur} on {user_id}: {reason}"


# Convenience list for binding
MODERATION_TOOLS = [get_server_rules, get_user_warn_history, issue_moderation_action]

## Summary

| Concept | Key takeaway |
|---------|-------------|
| `@tool` decorator | Easiest way to create a tool — uses function name, docstring, and type hints |
| Pydantic `args_schema` | Adds field descriptions and validation — the LLM reads these! |
| `.bind_tools()` | Tells the model which tools are available |
| `AIMessage.tool_calls` | Structured list of tool requests (name + args + ID) |
| `ToolMessage` | How you send tool results back to the model |
| Tool execution loop | Human → AI (tool call) → Execute → ToolMessage → AI (final answer) |
| Good descriptions | Tool and field descriptions are part of the prompt — be specific |

**Next up → Lesson 4:** The ReAct agent — `create_react_agent` automates the tool loop so you never write it manually again.